# ⚙️ Notebook 02 — Preprocesamiento & Feature Engineering
## EnergyForecast · IA EAFIT 2026-1

**Prerequisito:** Haber ejecutado `01_eda.ipynb` (genera `data/processed/PJME_clean.csv`)

Este notebook construye:
- Lag features (24h, 48h, 168h)
- Features de rolling statistics
- Encodings cíclicos (seno/coseno)
- Secuencias LSTM listas para entrenamiento

In [ ]:
# ── Fix Windows event loop (Python 3.12+) ────────────────────────────────────
import asyncio
import sys
if sys.platform == 'win32':
    asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())
print('✅ Event loop policy configurado')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import os
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

EAFIT_BLUE = '#003d79'
os.makedirs('data/processed', exist_ok=True)
os.makedirs('models/checkpoints', exist_ok=True)
print('✅ Imports OK')

## 1. Carga de datos limpios

In [ ]:
df = pd.read_csv('data/processed/PJME_clean.csv', index_col='Datetime', parse_dates=True)
print(f'Shape: {df.shape}')
df.head(3)

## 2. Lag features y rolling statistics

In [ ]:
# ── Lag features ──────────────────────────────────────────────────────────────
# Lag 24h: mismo momento ayer
# Lag 48h: anteayer
# Lag 168h: misma hora de la semana pasada (muy fuerte en ACF)

df_feat = df.copy()

lags = [1, 2, 3, 6, 12, 24, 48, 72, 168]
for lag in lags:
    df_feat[f'lag_{lag}h'] = df_feat['MW'].shift(lag)

# Rolling statistics (ventana de 24h y 168h)
df_feat['roll_mean_24h']  = df_feat['MW'].shift(1).rolling(24).mean()
df_feat['roll_std_24h']   = df_feat['MW'].shift(1).rolling(24).std()
df_feat['roll_mean_168h'] = df_feat['MW'].shift(1).rolling(168).mean()
df_feat['roll_max_24h']   = df_feat['MW'].shift(1).rolling(24).max()
df_feat['roll_min_24h']   = df_feat['MW'].shift(1).rolling(24).min()

print(f'Features totales: {df_feat.shape[1]}')
print(f'Nuevas features: {[c for c in df_feat.columns if c != "MW" and c not in df.columns]}')

In [ ]:
# ── Encodings cíclicos (para capturar periodicidad en XGBoost) ────────────────
# sin/cos para hora del día (periodo 24)
df_feat['hour_sin'] = np.sin(2 * np.pi * df_feat['hour'] / 24)
df_feat['hour_cos'] = np.cos(2 * np.pi * df_feat['hour'] / 24)

# sin/cos para día de la semana (periodo 7)
df_feat['dow_sin'] = np.sin(2 * np.pi * df_feat['dayofweek'] / 7)
df_feat['dow_cos'] = np.cos(2 * np.pi * df_feat['dayofweek'] / 7)

# sin/cos para mes (periodo 12)
df_feat['month_sin'] = np.sin(2 * np.pi * df_feat['month'] / 12)
df_feat['month_cos'] = np.cos(2 * np.pi * df_feat['month'] / 12)

# sin/cos para día del año (periodo 365)
df_feat['doy_sin'] = np.sin(2 * np.pi * df_feat['dayofyear'] / 365)
df_feat['doy_cos'] = np.cos(2 * np.pi * df_feat['dayofyear'] / 365)

print(f'✅ Encodings cíclicos agregados. Total features: {df_feat.shape[1]}')

In [ ]:
# ── Eliminar NaN generados por lags ───────────────────────────────────────────
print(f'Antes de dropna: {len(df_feat):,} filas')
df_feat = df_feat.dropna()
print(f'Después de dropna: {len(df_feat):,} filas')
print(f'Filas eliminadas por lags/rolling: {len(df) - len(df_feat):,} ({(len(df)-len(df_feat))/len(df)*100:.1f}%)')

## 3. Split temporal y escalado

In [ ]:
# ── Definir features para XGBoost ─────────────────────────────────────────────
FEATURE_COLS = [
    'hour', 'dayofweek', 'month', 'quarter', 'year', 'dayofyear', 'is_weekend',
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
    'month_sin', 'month_cos', 'doy_sin', 'doy_cos',
    'lag_1h', 'lag_2h', 'lag_3h', 'lag_6h', 'lag_12h',
    'lag_24h', 'lag_48h', 'lag_72h', 'lag_168h',
    'roll_mean_24h', 'roll_std_24h', 'roll_mean_168h',
    'roll_max_24h', 'roll_min_24h'
]
TARGET = 'MW'

# Split temporal (sin data leakage)
train = df_feat[:'2014']
val   = df_feat['2015':'2015']
test  = df_feat['2016':]

X_train, y_train = train[FEATURE_COLS], train[TARGET]
X_val,   y_val   = val[FEATURE_COLS],   val[TARGET]
X_test,  y_test  = test[FEATURE_COLS],  test[TARGET]

print('📦 Dataset para XGBoost:')
print(f'   X_train: {X_train.shape} | y_train: {y_train.shape}')
print(f'   X_val:   {X_val.shape}   | y_val:   {y_val.shape}')
print(f'   X_test:  {X_test.shape}  | y_test:  {y_test.shape}')

In [ ]:
# ── Escalado para LSTM ─────────────────────────────────────────────────────────
# LSTM es sensible a la escala → normalizamos MW entre 0 y 1 (MinMax)
from sklearn.preprocessing import MinMaxScaler

scaler_mw = MinMaxScaler(feature_range=(0, 1))
# Fit SOLO en train (no leakage)
scaler_mw.fit(df_feat[:'2014'][['MW']])

# Transformar toda la serie para crear secuencias
series_scaled = scaler_mw.transform(df_feat[['MW']]).flatten()

joblib.dump(scaler_mw, 'models/checkpoints/scaler_mw.pkl')
print(f'✅ Scaler guardado en models/checkpoints/scaler_mw.pkl')
print(f'   MW escalado → rango: [{series_scaled.min():.3f}, {series_scaled.max():.3f}]')

## 4. Creación de secuencias para LSTM

In [ ]:
# ── Función para crear secuencias deslizantes ─────────────────────────────────
def create_sequences(data: np.ndarray, window: int = 168, horizon: int = 24):
    """
    Crea secuencias de longitud `window` para predecir `horizon` pasos adelante.
    window=168 → 1 semana de contexto
    horizon=24 → predicción de las próximas 24 horas
    """
    X, y = [], []
    for i in range(len(data) - window - horizon + 1):
        X.append(data[i : i + window])
        y.append(data[i + window : i + window + horizon])
    return np.array(X), np.array(y)

WINDOW  = 168   # 1 semana de contexto
HORIZON = 24    # predice las próximas 24 horas

# Índices temporales para el split
idx_train_end = df_feat.index.searchsorted(pd.Timestamp('2015-01-01'))
idx_val_end   = df_feat.index.searchsorted(pd.Timestamp('2016-01-01'))

series_train = series_scaled[:idx_train_end]
series_val   = series_scaled[idx_train_end - WINDOW : idx_val_end]   # incluye contexto
series_test  = series_scaled[idx_val_end - WINDOW :]                  # incluye contexto

X_lstm_train, y_lstm_train = create_sequences(series_train, WINDOW, HORIZON)
X_lstm_val,   y_lstm_val   = create_sequences(series_val,   WINDOW, HORIZON)
X_lstm_test,  y_lstm_test  = create_sequences(series_test,  WINDOW, HORIZON)

# LSTM espera (batch, timesteps, features)
X_lstm_train = X_lstm_train[..., np.newaxis]
X_lstm_val   = X_lstm_val[..., np.newaxis]
X_lstm_test  = X_lstm_test[..., np.newaxis]

print('📦 Secuencias para LSTM:')
print(f'   X_train: {X_lstm_train.shape} | y_train: {y_lstm_train.shape}')
print(f'   X_val:   {X_lstm_val.shape}   | y_val:   {y_lstm_val.shape}')
print(f'   X_test:  {X_lstm_test.shape}  | y_test:  {y_lstm_test.shape}')

In [ ]:
# ── Guardar arrays y features list ────────────────────────────────────────────
np.save('data/processed/X_lstm_train.npy', X_lstm_train)
np.save('data/processed/y_lstm_train.npy', y_lstm_train)
np.save('data/processed/X_lstm_val.npy',   X_lstm_val)
np.save('data/processed/y_lstm_val.npy',   y_lstm_val)
np.save('data/processed/X_lstm_test.npy',  X_lstm_test)
np.save('data/processed/y_lstm_test.npy',  y_lstm_test)

# Guardar features y targets de XGBoost
X_train.to_csv('data/processed/X_train_xgb.csv')
X_val.to_csv('data/processed/X_val_xgb.csv')
X_test.to_csv('data/processed/X_test_xgb.csv')
y_train.to_csv('data/processed/y_train_xgb.csv')
y_val.to_csv('data/processed/y_val_xgb.csv')
y_test.to_csv('data/processed/y_test_xgb.csv')

# Guardar lista de features
import json
with open('data/processed/feature_cols.json', 'w') as f:
    json.dump(FEATURE_COLS, f)

print('✅ Todos los archivos guardados.')
print('\n   → Continúa con 03_modeling.ipynb')